# 第313章: SDEdit — 画像変容の実装

## 📋 この章で学ぶこと

- [ ] SDEditのアルゴリズムを実装できる（ノイズ付与→条件付きデノイズング）
- [ ] ノイズ付与量（t_start）が「変容の自由度」と「構造保持」のトレードオフを制御することを理解できる
- [ ] structure+textureの分離という観点でSDEditを解釈できる
- [ ] DDIM Inversionとの違いと使い分けを説明できる

## 🎯 前提知識

- ✅ Notebook 310-312（DDIM Inversion、ノイズ補間、条件補間）

⏱️ **推定学習時間**: 90-120分
📊 **難易度**: ★★★★☆（上級）
🎓 **カテゴリ**: 実践

---

## 🌟 はじめに

SDEdit（Song et al., 2022）は、シンプルで強力な画像編集手法です。

### SDEditのアルゴリズム

```
  入力画像（編集してほしい画像または粗い編集）
       │
       ▼ ① ノイズを付与（t_start までノイズ化）
  ノイズ画像 x_{t_start}
       │
       ▼ ② 条件付きDDIMサンプリング（t_start → 0）
  出力画像（構造を保持しつつ変容）
```

### DDIMInversion との比較

| 手法 | Inversion | SDEdit |
|------|-----------|--------|
| 入力→ノイズ | 決定的逆過程 | **確率的ノイズ付与** |
| 計算コスト | Inversionに時間 | **高速**（Inversionなし）|
| 構造保持 | ほぼ完璧 | t_start に依存 |
| 編集の自由度 | 限定的 | **高い** |

### t_start の役割（SDEditの核心）

```
  t_start=0:   ノイズなし → 変化なし（元画像のまま）
  t_start=T/4: 少しノイズ → 色やテクスチャの変更
  t_start=T/2: 中程度     → 構造を保持した意味的変容
  t_start=T:   完全ノイズ → 無関係な画像が生成される
```

### 📝 この章の構成

1. **SDEditの実装** — ノイズ付与→条件付きデノイズ
2. **t_start の探索** — 変容の粗さを制御
3. **異なる条件でのSDEdit** — 「3」を様々な「8系」に変容
4. **ストロークから画像生成** — スケッチ編集への応用
5. **DDIM Inversion vs SDEdit** — 品質とコストの比較

In [ ]:
# ============================================================
# 環境設定
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import warnings

warnings.filterwarnings('ignore')

def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'MS Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)
print(f"Device: {device}")
print("✅ 環境設定完了")

In [ ]:
# ============================================================
# 条件付き拡散モデルの定義と学習
# ============================================================

def get_schedule(n_steps=100, beta_start=1e-4, beta_end=0.02):
    betas = np.linspace(beta_start, beta_end, n_steps)
    alphas = 1.0 - betas
    alpha_bars = np.cumprod(alphas)
    return {'betas': torch.tensor(betas, dtype=torch.float32),
            'alphas': torch.tensor(alphas, dtype=torch.float32),
            'alpha_bars': torch.tensor(alpha_bars, dtype=torch.float32)}

class ConditionalNoisePredictor(nn.Module):
    def __init__(self, dim=784, n_classes=10, te_dim=32, ce_dim=32):
        super().__init__()
        self.null_token = nn.Parameter(torch.zeros(ce_dim))
        self.class_emb = nn.Embedding(n_classes, ce_dim)
        self.time_mlp = nn.Sequential(nn.Linear(1, te_dim), nn.SiLU(), nn.Linear(te_dim, te_dim))
        self.net = nn.Sequential(
            nn.Linear(dim + te_dim + ce_dim, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, dim),
        )

    def forward(self, x, t, c=None):
        t_emb = self.time_mlp(t.float().unsqueeze(-1) / 100.0)
        c_emb = self.null_token.unsqueeze(0).expand(x.shape[0], -1) if c is None else self.class_emb(c)
        return self.net(torch.cat([x, t_emb, c_emb], dim=-1))

    def forward_cfg(self, x, t, c, w=5.0):
        eps_u = self.forward(x, t, c=None)
        eps_c = self.forward(x, t, c=c)
        return eps_u + w * (eps_c - eps_u)

transform = transforms.ToTensor()
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

schedule = get_schedule(n_steps=100)
alpha_bars = schedule['alpha_bars'].to(device)

model = ConditionalNoisePredictor().to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-4)

print("条件付き拡散モデル学習中...")
for epoch in range(2):
    model.train()
    total = 0
    for x, y in train_loader:
        x = x.view(-1, 784).to(device) * 2 - 1
        y = y.to(device)
        t = torch.randint(0, 100, (x.shape[0],)).to(device)
        ab = alpha_bars[t].unsqueeze(-1)
        noise = torch.randn_like(x)
        x_noisy = torch.sqrt(ab) * x + torch.sqrt(1 - ab) * noise
        mask_u = torch.rand(x.shape[0], device=device) < 0.15
        t_emb = model.time_mlp(t.float().unsqueeze(-1) / 100.0)
        c_emb = torch.where(mask_u.unsqueeze(-1),
                            model.null_token.unsqueeze(0).expand(x.shape[0], -1),
                            model.class_emb(y))
        loss = nn.functional.mse_loss(model.net(torch.cat([x_noisy, t_emb, c_emb], dim=-1)), noise)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1}/15 | Loss: {total/len(train_loader):.6f}")

model.eval()
print("✅ 学習完了")

---

## 1. SDEditの実装

In [ ]:
# ============================================================
# SDEditアルゴリズムの実装
# ============================================================

@torch.no_grad()
def sdedit(model, x_0, target_class, t_start_frac=0.5, w=5.0, n_steps=5):
    """
    SDEdit: ① ノイズ付与 → ② 条件付きデノイズ
    t_start_frac: ノイズ付与の強さ (0=変化なし, 1=完全ノイズ)
    """
    ab = schedule['alpha_bars'].to(device)
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)

    # ① ノイズ付与（確率的）
    t_start_idx = int(t_start_frac * n_steps)
    t_start = step_idx[t_start_idx]
    ab_t = ab[t_start]
    noise = torch.randn_like(x_0)
    x_t = torch.sqrt(ab_t) * x_0 + torch.sqrt(1 - ab_t) * noise  # 正方向過程

    # ② DDIMで条件付きデノイズ（t_start → 0）
    c = torch.full((x_t.shape[0],), target_class, dtype=torch.long).to(device)
    x = x_t.clone()
    for i in range(t_start_idx, 0, -1):
        t_c, t_p = step_idx[i], step_idx[i - 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)
        eps = model.forward_cfg(x, t_t, c, w=w)
        x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
        x = torch.sqrt(ab[t_p]) * x0_p + torch.sqrt(1 - ab[t_p]) * eps

    return x

# テスト画像の取得
test_batch, test_labels = next(iter(test_loader))
test_imgs = test_batch.view(-1, 784).to(device) * 2 - 1

print("✅ SDEdit関数定義完了")
print("  使い方: sdedit(model, 入力画像, 目標クラス, t_start_frac=0.5)")

In [ ]:
# ============================================================
# t_start_frac の探索
# ============================================================

# 数字3を変様々なt_startで変容させる
idx3 = (test_labels == 3).nonzero()[0].item()
real_3 = test_imgs[idx3:idx3+1]

t_start_fracs = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
target_classes_t = [8, 8, 8, 8, 8, 8]

fig, axes = plt.subplots(2, len(t_start_fracs) + 1, figsize=(16, 4.5))

# 1行目: 3→8 変容
img_orig_3 = ((real_3[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
axes[0, 0].imshow(img_orig_3, cmap='gray')
axes[0, 0].set_title('元画像
(3)', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

for col, (t_frac, tc) in enumerate(zip(t_start_fracs, target_classes_t)):
    result = sdedit(model, real_3, tc, t_start_frac=t_frac, w=5.0)
    img_r = ((result[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
    axes[0, col + 1].imshow(img_r, cmap='gray')
    axes[0, col + 1].set_title(f't={t_frac:.1f}
→{tc}', fontsize=10)
    axes[0, col + 1].axis('off')

# 2行目: 3→1 変容
axes[1, 0].imshow(img_orig_3, cmap='gray')
axes[1, 0].set_title('元画像
(3)', fontsize=11, fontweight='bold')
axes[1, 0].axis('off')

for col, t_frac in enumerate(t_start_fracs):
    result = sdedit(model, real_3, 1, t_start_frac=t_frac, w=5.0)
    img_r = ((result[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
    axes[1, col + 1].imshow(img_r, cmap='gray')
    axes[1, col + 1].set_title(f't={t_frac:.1f}
→1', fontsize=10)
    axes[1, col + 1].axis('off')

fig.suptitle('SDEdit: t_start_fracの影響（上: →8, 下: →1）
小さいt_start=構造保持, 大きいt_start=自由な変容',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_313_01_t_start_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 t=0.2: 元の3の構造がほぼそのまま変容")
print("   t=0.9: 8や1の特徴が強く表れ、3らしさが薄れる")

In [ ]:
# ============================================================
# 様々な対象クラスへの変容
# ============================================================

# 数字5を様々なクラスに変容
idx5 = (test_labels == 5).nonzero()[0].item()
real_5 = test_imgs[idx5:idx5+1]
img_orig_5 = ((real_5[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)

target_digits = list(range(10))
t_frac_use = 0.5

fig, axes = plt.subplots(1, 11, figsize=(18, 2.5))
axes[0].imshow(img_orig_5, cmap='gray')
axes[0].set_title('元画像(5)', fontsize=10, fontweight='bold')
axes[0].axis('off')

for col, td in enumerate(target_digits):
    result = sdedit(model, real_5, td, t_start_frac=t_frac_use, w=5.0)
    img_r = ((result[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
    axes[col + 1].imshow(img_r, cmap='gray')
    axes[col + 1].set_title(f'→{td}', fontsize=10)
    axes[col + 1].axis('off')

fig.suptitle(f'SDEdit: 様々なクラスへの変容（t_start={t_frac_use}）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_313_02_multi_target.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 2. スケッチから画像へ（ストローク編集のシミュレーション）

SDEditの有名なユースケース：粗い編集（スケッチ）→ リアルな画像

```
  ユーザーのスケッチ → SDEdit → フォトリアルな画像
  （粗い輪郭のみ）          （テクスチャ、色が自然に補完）
```

In [ ]:
# ============================================================
# スケッチ→画像変換のシミュレーション
# 元の数字を「劣化」させてスケッチ化し、SDEditで復元
# ============================================================

def make_sketch(img_tensor):
    """数字画像を「スケッチ風」に劣化させる"""
    img_np = ((img_tensor.cpu().numpy().reshape(28, 28) + 1) / 2)
    # エッジ検出風の劣化: ガウシアンブラー後に二値化
    from scipy.ndimage import gaussian_filter
    blurred = gaussian_filter(img_np, sigma=1.5)
    edges = np.abs(np.gradient(blurred)[0]) + np.abs(np.gradient(blurred)[1])
    sketch = (edges > edges.mean() * 0.5).astype(float)
    # ランダムな線を追加（手書き風）
    sketch_noise = sketch + np.random.randn(*sketch.shape) * 0.1
    return np.clip(sketch_noise, 0, 1)

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
demo_digits = [0, 1, 3, 5, 7, 9]

for col, d in enumerate(demo_digits):
    idx_d = (test_labels == d).nonzero()[0].item()
    real_img = test_imgs[idx_d:idx_d+1]

    # 元画像
    img_o = ((real_img[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
    axes[0, col].imshow(img_o, cmap='gray')
    axes[0, col].set_title(f'元({d})', fontsize=11)
    axes[0, col].axis('off')

    # スケッチ化
    sketch_np = make_sketch(real_img[0])
    axes[1, col].imshow(sketch_np, cmap='gray')
    axes[1, col].set_title('スケッチ', fontsize=11)
    axes[1, col].axis('off')

    # SDEdit（スケッチ→自然な数字へ）
    sketch_tensor = torch.tensor(sketch_np * 2 - 1, dtype=torch.float32).unsqueeze(0).to(device)
    result = sdedit(model, sketch_tensor.view(1, 784), d, t_start_frac=0.4, w=5.0)
    img_r = ((result[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
    axes[2, col].imshow(img_r, cmap='gray')
    axes[2, col].set_title('SDEdit復元', fontsize=11)
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('元画像', fontsize=11, rotation=0, labelpad=45)
axes[1, 0].set_ylabel('スケッチ', fontsize=11, rotation=0, labelpad=45)
axes[2, 0].set_ylabel('SDEdit復元', fontsize=11, rotation=0, labelpad=45)

fig.suptitle('スケッチ→リアル画像変換
（SDEditでエッジのみから自然な数字を生成）',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_313_03_sketch_to_image.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# SDEdit vs DDIM Inversion の比較
# ============================================================

idx3 = (test_labels == 3).nonzero()[0].item()
real_3 = test_imgs[idx3:idx3+1]

# SDEdit（高速、ストカスティック）
@torch.no_grad()
def ddim_inversion_fn(model, x_0, class_label, w=1.0, n_steps=5):
    ab = schedule['alpha_bars'].to(device)
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)
    c = torch.full((x_0.shape[0],), class_label, dtype=torch.long).to(device)
    x = x_0.clone()
    for i in range(n_steps):
        t_c, t_n = step_idx[i], step_idx[i + 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)
        eps = model.forward_cfg(x, t_t, c, w=w)
        x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
        x = torch.sqrt(ab[t_n]) * x0_p + torch.sqrt(1 - ab[t_n]) * eps
    return x

@torch.no_grad()
def ddim_sample_fn(model, x_T, class_label, w=5.0, n_steps=5):
    ab = schedule['alpha_bars'].to(device)
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)
    c = torch.full((x_T.shape[0],), class_label, dtype=torch.long).to(device)
    x = x_T.clone()
    for i in range(n_steps, 0, -1):
        t_c, t_p = step_idx[i], step_idx[i - 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)
        eps = model.forward_cfg(x, t_t, c, w=w)
        x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
        x = torch.sqrt(ab[t_p]) * x0_p + torch.sqrt(1 - ab[t_p]) * eps
    return x

target = 8

# SDEdit（複数t_start）
sdedit_results = {}
for t_frac in [0.3, 0.5, 0.7]:
    sdedit_results[t_frac] = sdedit(model, real_3, target, t_start_frac=t_frac, w=5.0)

# DDIM Inversion + Sampling（条件変更）
noise_3 = ddim_inversion_fn(model, real_3, class_label=3, w=1.0, n_steps=5)
inv_result = ddim_sample_fn(model, noise_3, target, w=5.0, n_steps=5)

fig, axes = plt.subplots(1, 6, figsize=(14, 2.5))
img_orig = ((real_3[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
axes[0].imshow(img_orig, cmap='gray'); axes[0].set_title('元画像(3)', fontsize=10); axes[0].axis('off')

for col, (t_frac, result) in enumerate(sdedit_results.items()):
    img_r = ((result[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
    axes[col + 1].imshow(img_r, cmap='gray')
    axes[col + 1].set_title(f'SDEdit
t={t_frac}', fontsize=10)
    axes[col + 1].axis('off')

inv_img = ((inv_result[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
axes[5].imshow(inv_img, cmap='gray')
axes[5].set_title('DDIM inv
+条件変更', fontsize=10)
axes[5].axis('off')

fig.suptitle(f'SDEdit vs DDIM Inversion（3→{target}）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_313_04_sdedit_vs_inversion.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 SDEdit:         高速・確率的・構造保持と自由度のバランスをt_startで制御")
print("   DDIM Inversion: 遅い・決定的・高精度な構造保持が可能")

---

## まとめ

### 🎯 学んだこと

**SDEditの仕組み**
- ✓ ノイズ付与（正過程）→ 条件付きデノイズ（逆過程）のシンプルな2段階
- ✓ t_startが「変容自由度」と「元構造の保持」を制御する唯一のハイパーパラメータ

**SDEdit vs DDIM Inversion**
- ✓ SDEdit: 高速・確率的・スケッチ→画像に最適
- ✓ DDIM Inversion: 高精度・決定的・実画像の精密編集に最適

**スケッチ→画像**
- ✓ 粗い入力（スケッチ、マスク）からSDEditで自然な画像を生成できる

### ⚠️ よくある間違い

❌ 「t_startを大きくすると品質が向上する」
✅ 大きすぎると元画像の情報が消えて条件から独立した画像が生成される

### ✅ 学習チェックリスト

- [ ] SDEditの2段階（ノイズ付与→デノイズ）を実装から説明できるか？
- [ ] t_startのトレードオフを具体例で説明できるか？
- [ ] SDEditとDDIM Inversionの使い分けを3つの観点で説明できるか？

---

## 🎓 自己評価クイズ

### Q1: SDEditとDDIM Inversionはどちらも「画像を変容」させるが、 t_start=1.0のSDeditはDDIM Inversionと同じか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 異なる。DDIM Inversionは決定的で元画像を高精度保存するが、t_start=1.0のSDEditは確率的ノイズ付与で元画像の情報をほぼ失う

SDEditのt_start=1.0は完全なランダムノイズから条件付き生成するのと等価で、元画像との関連がない。DDIM Inversionは同じxₜから同じx₀が再現できる。

</details>

---

### Q2: SDEditが「スケッチ→画像」に適している理由は？

<details>
<summary>💡 答えを見る</summary>

**答え**: スケッチ（エッジ情報）にノイズを付与してからデノイズすることで、スケッチの形状を保持しつつ自然なテクスチアを生成できるため

t_start を適切に選ぶと、スケッチの構造（輪郭）は保持されながら、テクスチャや細部が拡散モデルによって自然に補完される。これがDDIM Inversionより向いている。

</details>